# EXPLAIN and Query Plans

Understanding **how Spark executes a query before it runs** is the foundation of performance tuning.

`EXPLAIN` surfaces the **query plan** — a tree of operations showing exactly what Spark will do:
which files it will read, which filters and projections it will push down, what physical operators it will use, and whether Photon or AQE will be active.

Every plan passes through four stages inside Spark's **Catalyst optimizer**:

| Stage | What it represents |
|---|---|
| **Parsed Logical Plan** | Raw AST — column names are unresolved strings |
| **Analyzed Logical Plan** | References resolved against the catalog; types validated |
| **Optimized Logical Plan** | Catalyst rules applied: column pruning, predicate pushdown, constant folding, … |
| **Physical Plan** | Concrete operators selected (HashAggregate, SortMergeJoin, FileScan …) |

> This notebook uses two dedicated Delta tables in `population_metrics.default` created in the Setup section.

## Setup

In [1]:
import os, sys
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(f"Wrong kernel: {sys.executable} — select .venv in VSCode")
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

+--------------------------------------+----------------+
|current_catalog()                     |current_schema()|
+--------------------------------------+----------------+
|dbw_dataengineering_sandbox_euwest_001|default         |
+--------------------------------------+----------------+



## Demo Dataset

Two Delta tables are used throughout this notebook:

| Table | Rows | Purpose |
|---|---|---|
| `explain_sales` | 10 | Wide fact table (14 columns) — ideal for demonstrating column pruning and predicate pushdown |
| `explain_products` | 5 | Narrow dimension table — ideal for demonstrating broadcast joins |

In [ ]:
# ── explain_sales ────────────────────────────────────────────────────────────
try:
    sales_count = spark.table("population_metrics.default.explain_sales").count()
except Exception:
    sales_count = 0

if sales_count == 0:
    sales_data = [
        (1,  1, 101, "EMEA", "Electronics", 1200.00, 1, 0.05, 120.00, "2024-01-10", "completed", "EUR", "card",   ""),
        (2,  2, 102, "AMER", "Electronics",  850.00, 2, 0.10,  85.00, "2024-01-11", "completed", "USD", "paypal", ""),
        (3,  3, 103, "APAC", "Apparel",       320.00, 3, 0.00,  32.00, "2024-01-12", "pending",   "JPY", "card",   "rush order"),
        (4,  1, 104, "EMEA", "Electronics", 1200.00, 1, 0.15, 120.00, "2024-01-13", "completed", "GBP", "bank",   ""),
        (5,  4, 105, "AMER", "Apparel",        75.00, 4, 0.00,   7.50, "2024-01-14", "completed", "USD", "card",   ""),
        (6,  2, 106, "EMEA", "Electronics",  850.00, 1, 0.05,  85.00, "2024-01-15", "refunded",  "EUR", "paypal", "customer return"),
        (7,  5, 107, "APAC", "Home",          200.00, 2, 0.10,  20.00, "2024-01-16", "completed", "AUD", "card",   ""),
        (8,  3, 108, "AMER", "Apparel",       320.00, 1, 0.00,  32.00, "2024-01-17", "completed", "USD", "card",   ""),
        (9,  4, 109, "EMEA", "Apparel",        75.00, 6, 0.20,   7.50, "2024-01-18", "completed", "EUR", "card",   "bulk order"),
        (10, 5, 110, "APAC", "Home",          200.00, 1, 0.00,  20.00, "2024-01-19", "pending",   "SGD", "bank",   ""),
    ]
    schema = (
        "sale_id INT, product_id INT, customer_id INT, region STRING, category STRING, "
        "amount DOUBLE, quantity INT, discount_pct DOUBLE, tax_amount DOUBLE, "
        "created_date STRING, status STRING, currency STRING, payment_method STRING, notes STRING"
    )
    (
        spark.createDataFrame(sales_data, schema=schema)
             .withColumn("created_date", F.to_date("created_date"))
             .write.format("delta").mode("overwrite")
             .saveAsTable("population_metrics.default.explain_sales")
    )
    print("explain_sales created.")
else:
    print(f"explain_sales already has {sales_count} rows — skipping.")

# ── explain_products ─────────────────────────────────────────────────────────
try:
    prod_count = spark.table("population_metrics.default.explain_products").count()
except Exception:
    prod_count = 0

if prod_count == 0:
    products_data = [
        (1, "UltraPhone X",  "Electronics",  999.99, "TechCorp"),
        (2, "ProLaptop 15",  "Electronics",  799.99, "TechCorp"),
        (3, "Classic Tee",   "Apparel",        19.99, "FashionCo"),
        (4, "Slim Chinos",   "Apparel",        49.99, "FashionCo"),
        (5, "Smart Speaker", "Home",          149.99, "HomeHub"),
    ]
    prod_schema = "product_id INT, product_name STRING, category STRING, list_price DOUBLE, brand STRING"
    (
        spark.createDataFrame(products_data, schema=prod_schema)
             .write.format("delta").mode("overwrite")
             .saveAsTable("population_metrics.default.explain_products")
    )
    print("explain_products created.")
else:
    print(f"explain_products already has {prod_count} rows — skipping.")

spark.table("population_metrics.default.explain_sales").show(truncate=False)

explain_sales created.
explain_products created.
+-------+----------+-----------+------+-----------+------+--------+------------+----------+------------+---------+--------+--------------+---------------+
|sale_id|product_id|customer_id|region|category   |amount|quantity|discount_pct|tax_amount|created_date|status   |currency|payment_method|notes          |
+-------+----------+-----------+------+-----------+------+--------+------------+----------+------------+---------+--------+--------------+---------------+
|1      |1         |101        |EMEA  |Electronics|1200.0|1       |0.05        |120.0     |2024-01-10  |completed|EUR     |card          |               |
|2      |2         |102        |AMER  |Electronics|850.0 |2       |0.1         |85.0      |2024-01-11  |completed|USD     |paypal        |               |
|3      |3         |103        |APAC  |Apparel    |320.0 |3       |0.0         |32.0      |2024-01-12  |pending  |JPY     |card          |rush order     |
|4      |1         |1

## 1. EXPLAIN Syntax — SQL and Python

**SQL syntax** — prefix any SELECT with `EXPLAIN`:

```sql
EXPLAIN SELECT region, SUM(amount) FROM explain_sales GROUP BY region
EXPLAIN EXTENDED  SELECT ...
EXPLAIN FORMATTED SELECT ...
EXPLAIN COST      SELECT ...
EXPLAIN CODEGEN   SELECT ...
```

**Python syntax** — call `.explain()` on any DataFrame:

```python
df.explain()                    # default: physical plan only
df.explain(mode="simple")       # same as above
df.explain(mode="extended")     # all 4 plan stages
df.explain(mode="cost")         # optimized plan + row/column statistics
df.explain(mode="formatted")    # formatted physical plan (easier to read)
df.explain(mode="codegen")      # physical plan + generated Java bytecode
```

> `df.explain()` prints directly to stdout and returns `None`.  
> For SQL `EXPLAIN`, the plan text is returned as a single-column DataFrame — collect it to print.

In [ ]:
# SQL EXPLAIN — plan text comes back as a DataFrame row
plan_text = spark.sql("""
    EXPLAIN
    SELECT region, SUM(amount) AS total
    FROM   population_metrics.default.explain_sales
    GROUP  BY region
""").collect()[0][0]

print(plan_text)

In [ ]:
# Python .explain() — same query, same plan, shorter syntax
(
    spark.table("population_metrics.default.explain_sales")
         .groupBy("region")
         .agg(F.sum("amount").alias("total"))
         .explain()       # prints physical plan
)

## 2. EXPLAIN Modes

| Mode | What you see | When to use |
|---|---|---|
| `simple` | Physical plan only | Quick check — is this a scan, join, aggregate? |
| `extended` | All 4 stages (Parsed → Analyzed → Optimized → Physical) | Debugging: compare what you wrote vs what Catalyst produced |
| `cost` | Optimized logical plan with row count / size estimates | Check whether the optimizer has useful statistics |
| `formatted` | Physical plan formatted with indentation and node IDs | Reading complex plans — multi-join pipelines |
| `codegen` | Physical plan + generated Java code per stage | Advanced: check whether whole-stage code gen fired |

The most useful modes day-to-day are **`extended`** (to see all optimisations) and **`formatted`** (to read complex physical plans).

In [ ]:
df = (
    spark.table("population_metrics.default.explain_sales")
         .filter(F.col("region") == "EMEA")
         .select("sale_id", "region", "amount")
)

print("=" * 70)
print("MODE: simple  — physical plan only")
print("=" * 70)
df.explain(mode="simple")

print("\n" + "=" * 70)
print("MODE: formatted  — annotated physical plan (easier to read for complex DAGs)")
print("=" * 70)
df.explain(mode="formatted")

## 3. Anatomy of a Query Plan

Plans are read **bottom-up**: the leaf node at the bottom produces data that flows upward through operators.

```
== Physical Plan ==
(3) Project [sale_id, region, amount]          ← 3. apply projection
+- (2) Filter (region = EMEA)                  ← 2. apply filter
   +- (1) Scan delta [sale_id,region,amount]   ← 1. read only needed columns from storage
         PushedFilters: [EqualTo(region,EMEA)]
```

Key things to look for:

| What you see in the plan | What it means |
|---|---|
| `Scan delta [...][col1, col2]` | Only listed columns are read from Parquet files |
| `PushedFilters: [EqualTo(...)]` | Filter is evaluated inside the file reader — rows skipped before entering Spark |
| `BroadcastHashJoin` | One side fits in memory and is broadcast to all executors |
| `SortMergeJoin` | Both sides are sorted and merged — used for large tables |
| `HashAggregate` | In-memory hash table aggregation |
| `Exchange` | Data shuffle across the network (expensive!) |
| `AdaptiveSparkPlan` | AQE wrapper — plan may change at runtime |
| `Photon*` prefix | Operator runs in Databricks' Photon vectorized engine |

## 4. All Four Plan Stages — `extended` Mode

The `extended` mode shows every stage Catalyst goes through.  
The gap between the **Analyzed** and **Optimized** plans is where Catalyst earns its keep.

In [ ]:
df = (
    spark.table("population_metrics.default.explain_sales")
         .filter(F.col("status") == "completed")
         .select("sale_id", "region", "amount", "status")
)

df.explain(mode="extended")

# What to compare between stages:
#
# Parsed Logical Plan
#   Project ['sale_id, 'region, 'amount, 'status]    ← unresolved (tick-prefixed)
#   +- Filter ('status = completed)
#      +- UnresolvedRelation [explain_sales]
#
# Analyzed Logical Plan
#   Project [sale_id#0, region#3, amount#5, status#10]  ← resolved with slot IDs
#   +- Filter (status#10 = completed)
#      +- Relation [sale_id#0, ..., notes#13] delta    ← ALL columns still listed
#
# Optimized Logical Plan
#   Project [sale_id#0, region#3, amount#5, status#10]
#   +- Filter (isnotnull(status#10) AND (status#10 = completed))
#      +- Relation [sale_id#0, region#3, amount#5, status#10] delta  ← only 4 columns!
#         (column pruning applied — 10 unused columns dropped from the scan)
#
# Physical Plan
#   Project / Filter / Scan delta ... PushedFilters: [IsNotNull(status), EqualTo(status,completed)]

## 5. Column Pruning

**Column pruning** is one of the most impactful Catalyst optimisations for wide tables.  
The optimizer walks the logical plan, determines which columns are actually referenced, and eliminates all others from the scan.

Because Delta Lake stores data in **Parquet** (a columnar format), an unreferenced column's file is never opened.  
On a table with 100 columns where your query touches 5, you may read **95 % less data** at zero code cost.

**How to spot it in the plan:**
- In the **Analyzed** plan: `Relation [all 14 columns] delta`
- In the **Optimized** plan: `Relation [col_a, col_b, col_c] delta` — only the referenced columns remain
- In the **Physical** plan: `Scan delta [col_a, col_b, col_c]` — the file reader is told to skip the rest

In [ ]:
print("=" * 70)
print("SELECT *  — all 14 columns pass through every stage")
print("=" * 70)
spark.table("population_metrics.default.explain_sales").explain(mode="extended")

print("\n\n" + "=" * 70)
print("SELECT 2 columns  — column pruning reduces the Relation to 2 columns")
print("=" * 70)
(
    spark.table("population_metrics.default.explain_sales")
         .select("amount", "region")
         .explain(mode="extended")
)

# In the Optimized Logical Plan for the second query:
#   Relation [amount#5, region#3] delta
# Only 2 Parquet column files are opened — 12 are skipped entirely.

## 6. Predicate Pushdown

**Predicate pushdown** moves filter conditions as far down the plan as possible — ideally into the storage layer itself.

For Delta Lake on Databricks this means:

| Level | Mechanism | Effect |
|---|---|---|
| **File skipping** | Delta's min/max statistics per file | Entire Parquet files skipped without opening |
| **Row group filtering** | Parquet row group statistics | Row groups within a file skipped |
| **Row-level filtering** | Pushed filter evaluated in the Parquet reader | Rows discarded before entering the Spark engine |

**How to spot it in the physical plan:**
```
Scan delta [region, amount]
    PushedFilters: [IsNotNull(region), EqualTo(region,EMEA)]
```
The `PushedFilters` line confirms the predicate reached the reader.
A separate `Filter` node above the scan means it was **not** pushed — Spark filters rows after they enter memory.

In [ ]:
print("=" * 70)
print("No filter — full scan")
print("=" * 70)
spark.table("population_metrics.default.explain_sales").explain(mode="simple")

print("\n" + "=" * 70)
print("With filter on 'region' — predicate pushed into the scan")
print("=" * 70)
(
    spark.table("population_metrics.default.explain_sales")
         .filter(F.col("region") == "EMEA")
         .explain(mode="simple")
)
# Look for: PushedFilters: [IsNotNull(region), EqualTo(region,EMEA)]

print("\n" + "=" * 70)
print("Filter on a computed column — cannot be pushed (no PushedFilters line)")
print("=" * 70)
(
    spark.table("population_metrics.default.explain_sales")
         .withColumn("net_amount", F.col("amount") * (1 - F.col("discount_pct")))
         .filter(F.col("net_amount") > 500)
         .explain(mode="simple")
)
# net_amount does not exist in storage — filter stays as a separate node above the scan

## 7. What the Catalyst Optimizer Does Between Plans

Between the **Analyzed** and **Optimized** logical plans, Catalyst applies a library of rules.  
The most common ones you will observe:

| Rule | What it does | Example |
|---|---|---|
| **ColumnPruning** | Remove unused columns from relations | `SELECT a, b FROM wide_table` — only a, b scanned |
| **PushDownPredicate** | Move filters toward data sources | `filter(col > 5)` appears inside `Scan` as `PushedFilters` |
| **ConstantFolding** | Evaluate constant expressions at plan time | `1 + 1` → `2`, `CAST(3 AS INT)` → `3` |
| **BooleanSimplification** | Simplify boolean logic | `x AND true` → `x`, `x OR false` → `x` |
| **CombineFilters** | Merge consecutive `.filter()` calls | `.filter(a).filter(b)` → single `Filter(a AND b)` |
| **EliminateSorts** | Remove sorts whose output is immediately re-sorted | Chained `orderBy` calls |
| **ReorderJoin** | Reorder joins based on table sizes | Small table moved to build side of hash join |

In [ ]:
# Demonstrate several Catalyst rules firing at once
df = (
    spark.table("population_metrics.default.explain_sales")
         # CombineFilters: two separate .filter() calls → one Filter node
         .filter(F.col("region") == "EMEA")
         .filter(F.col("status") == "completed")
         # ConstantFolding: literal expression resolved at plan time
         .withColumn("vat_rate", F.lit(0.20))
         # ColumnPruning: only 4 columns referenced
         .select("sale_id", "region", "amount", "vat_rate")
)

df.explain(mode="extended")

# What to observe in Optimized Logical Plan:
#  - Single Filter node combining both predicates (CombineFilters)
#  - Relation lists only [sale_id, region, amount, status] — vat_rate is a literal, not a column (ColumnPruning)
#  - isnotnull checks added automatically (NullPropagation)

## 8. Photon — Databricks' Vectorized Execution Engine

**Photon** is a native C++ execution engine built into Databricks that replaces the default JVM-based Spark operators for compatible operations.

### How Photon works

Standard Spark processes one row at a time through the JVM.  
Photon processes **batches of columns** using SIMD (Single Instruction, Multiple Data) CPU instructions — the same technique used by high-performance analytical databases like DuckDB.

| Aspect | Standard Spark (JVM) | Photon (C++) |
|---|---|---|
| Processing model | Row-at-a-time | Columnar batch (vectorized) |
| Memory format | JVM objects | Apache Arrow columnar buffers |
| GC pressure | High (many objects) | Low (off-heap native memory) |
| CPU utilisation | Limited by JVM overhead | SIMD / cache-efficient |
| Availability | All Databricks runtimes | DBR 9.1+ on supported cluster types |

### How to identify Photon in EXPLAIN output

When Photon is active, physical plan operators carry a `Photon` prefix:

| Standard operator | Photon equivalent |
|---|---|
| `FileScan delta` | `PhotonScan delta` |
| `Filter` | `PhotonFilter` |
| `Project` | `PhotonProject` |
| `HashAggregate` | `PhotonGroupingAgg` |
| `Exchange` (shuffle) | `PhotonShuffleExchangeExec` |
| `SortMergeJoin` | `PhotonSortMergeJoin` |
| `BroadcastHashJoin` | `PhotonBroadcastHashJoin` |

If an operator is **not yet supported by Photon** (e.g., certain UDFs, complex types), it falls back transparently to the standard JVM operator — you will see both `Photon*` and standard nodes in the same plan.

> **Tip:** Photon is enabled by default on most Databricks cluster types. Check `spark.databricks.photon.enabled` to confirm.

In [ ]:
# Check whether Photon is enabled on this cluster
photon_enabled = spark.conf.get("spark.databricks.photon.enabled", "not set")
print(f"spark.databricks.photon.enabled = {photon_enabled}")

print()

# Run an aggregation — this is a typical Photon workload
# On a Photon cluster the physical plan will show PhotonGroupingAgg, PhotonScan, etc.
# On a standard cluster you will see HashAggregate, FileScan, etc.
df = (
    spark.table("population_metrics.default.explain_sales")
         .groupBy("region", "category")
         .agg(
             F.count("*").alias("tx_count"),
             F.sum("amount").alias("total_amount"),
             F.avg("amount").alias("avg_amount"),
         )
)

print("Physical plan (look for Photon* operators if Photon is enabled):")
df.explain(mode="formatted")

## 9. Broadcast Hash Join

When one side of a join is small enough to fit in memory, Spark can **broadcast** it to every executor instead of shuffling both sides across the network.

### How it works

```
SortMergeJoin (default for large tables)
   shuffle both sides → sort → merge  — expensive (2 shuffles)

BroadcastHashJoin (when one side is small)
   build hash table from small side → broadcast to all executors
   probe hash table per partition of the large side — no shuffle!
```

### When Spark uses it automatically

Spark broadcasts a table when its estimated size is below `spark.sql.autoBroadcastJoinThreshold` (default **10 MB**).  
This threshold uses statistics from the Delta table — `ANALYZE TABLE` populates them.

### Forcing it with a hint

```python
from pyspark.sql.functions import broadcast
df_joined = large_df.join(broadcast(small_df), on="key")
```

In SQL:
```sql
SELECT /*+ BROADCAST(p) */ *
FROM explain_sales s JOIN explain_products p ON s.product_id = p.product_id
```

In [ ]:
sales    = spark.table("population_metrics.default.explain_sales")
products = spark.table("population_metrics.default.explain_products")  # 5 rows — tiny

print("=" * 70)
print("Without broadcast hint — Spark decides based on statistics")
print("(may already choose BroadcastHashJoin if stats show products is tiny)")
print("=" * 70)
sales.join(products, on="product_id").explain(mode="formatted")

print("\n" + "=" * 70)
print("With broadcast() hint — BroadcastHashJoin guaranteed")
print("=" * 70)
sales.join(broadcast(products), on="product_id").explain(mode="formatted")

# In the plan you will see:
#   BroadcastHashJoin [product_id], [product_id], Inner, BuildRight
#   +- BroadcastExchange HashedRelationBroadcastMode(...)  ← products side broadcasted

# Check the current threshold
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"\nautoBroadcastJoinThreshold = {threshold} bytes ({int(threshold)//1024//1024} MB)")

## 10. Adaptive Query Execution (AQE)

The Catalyst optimizer works at **plan time** — before any data is read.  
Sometimes the statistics it uses are wrong (outdated, missing, or based on estimates).

**AQE** (introduced in Spark 3.0, enabled by default in Databricks) re-optimises the plan at **runtime**, after each shuffle stage completes and real data sizes are known.

### What AQE can change at runtime

| Feature | What it does |
|---|---|
| **Coalesce shuffle partitions** | Merges small post-shuffle partitions to avoid many tiny tasks (e.g. 200 → 4 partitions) |
| **Switch join strategy** | Downgrades `SortMergeJoin` → `BroadcastHashJoin` if one side turns out to be small after filtering |
| **Skew join handling** | Splits oversized partitions from skewed keys into multiple sub-tasks |

### How to spot AQE in EXPLAIN output

`EXPLAIN` shows the **initial** plan before execution.  
When AQE is active, the physical plan is wrapped in:
```
AdaptiveSparkPlan (isFinalPlan=false)
```
After the query runs, `isFinalPlan=true` and the plan shows what actually executed (including any runtime changes).

In [ ]:
# ── AQE settings ─────────────────────────────────────────────────────────────
aqe_settings = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
]
print("Current AQE settings:")
for s in aqe_settings:
    print(f"  {s} = {spark.conf.get(s, 'not set')}")

print()

# ── See AdaptiveSparkPlan in the plan ─────────────────────────────────────────
df = (
    spark.table("population_metrics.default.explain_sales")
         .groupBy("region")
         .agg(F.sum("amount").alias("total"))
         .orderBy(F.desc("total"))
)

print("Plan before execution (AQE wrapper visible):")
df.explain(mode="formatted")
# Look for: AdaptiveSparkPlan isFinalPlan=false
# After df.show() / df.collect(), isFinalPlan becomes true
# and the plan reflects the actual execution (e.g. fewer partitions after coalescing)

In [ ]:
# Execute the query so AQE can finalise the plan
df.show(truncate=False)

# Now explain again — isFinalPlan=true, plan shows what actually ran
print("\nPlan AFTER execution (AQE final plan):")
df.explain(mode="formatted")
# Compare the number of shuffle partitions before vs after —
# AQE likely coalesced 200 default partitions down to 1 or 2 for this small dataset

## Summary — EXPLAIN Quick Reference

### Commands

```python
df.explain()                    # physical plan
df.explain(mode="extended")     # all 4 stages — use for debugging optimisations
df.explain(mode="cost")         # optimized plan + statistics
df.explain(mode="formatted")    # formatted physical plan — use for complex joins

spark.sql("EXPLAIN SELECT ...").collect()[0][0]   # SQL equivalent
```

### What to look for

| Signal in the plan | Good or bad? | Action |
|---|---|---|
| `Scan [few columns]` | Good | Column pruning is working |
| `Scan [all columns]` | Investigate | Are you really using all columns? |
| `PushedFilters: [...]` | Good | Predicate reached the reader |
| `Filter` node above `Scan` (no PushedFilters) | Investigate | Computed column filter — unavoidable, but be aware |
| `BroadcastHashJoin` | Good | No shuffle on the small side |
| `SortMergeJoin` | Neutral | Both sides are large — expected |
| `Exchange` | Watch | Each Exchange is a shuffle — minimise them |
| `AdaptiveSparkPlan isFinalPlan=false` | Good | AQE will optimise at runtime |
| `Photon*` prefix | Good | Vectorized C++ engine active |
| Standard operator with no `Photon*` in an otherwise Photon plan | Neutral | Fallback to JVM for that operator (UDF, unsupported type) |

### Column pruning — the single cheapest optimisation

```python
# ALWAYS select only the columns you need
df = spark.table("wide_table").select("a", "b", "c")   # 3 Parquet files opened
# vs
df = spark.table("wide_table")                          # ALL Parquet files opened
```

Catalyst applies column pruning automatically — but only for the columns referenced in your final output.  
Calling `.select("*")` or keeping the full DataFrame through many transformations prevents pruning.

## (Optional) Cleanup

In [ ]:
# Uncomment to remove the demo tables
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.explain_sales")
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.explain_products")
# print("Tables dropped.")